# A. Smoking Health Risk Prediction (Google Colab)

Notebook ini menjalankan pipeline machine learning end-to-end untuk memprediksi **risiko penyakit jantung koroner (CHD) dalam 10 tahun** berdasarkan data kesehatan dan kebiasaan merokok.

Sumber dataset: **Framingham Heart Study** (4.240 partisipan, 15 fitur + target `TenYearCHD`).

Target klasifikasi biner:
- **0** = risiko rendah (tidak mengalami CHD dalam 10 tahun)
- **1** = risiko tinggi (mengalami CHD dalam 10 tahun)

> **Disclaimer:** Hasil model adalah **estimasi statistik**, bukan diagnosis medis.

Model yang dibandingkan: **Logistic Regression**, **Random Forest**, dan **Gradient Boosting**. Model terbaik dipilih berdasarkan **ROC-AUC** pada test set karena dataset bersifat *imbalanced*.

# B. Setup & Install Dependencies

In [ ]:
# Colab umumnya sudah memiliki library inti; install jika diperlukan
# (jika lingkungan Google Colab terkini: sklearn, pandas, matplotlib, seaborn sudah tersedia)
!pip install -q pandas numpy matplotlib seaborn scikit-learn joblib

# C. Koneksi ke Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## C1. Konfigurasi path dataset

Ubah `DATA_PATH` di bawah ini jika lokasi file `framingham.csv` di Google Drive Anda berbeda. Semua output (model & hasil prediksi) disimpan ke `OUTPUT_DIR`.

In [ ]:
import os

# ==== UBAH BAGIAN INI SESUAI LOKASI DATASET ANDA ====
DATA_PATH = '/content/drive/MyDrive/smoking-health-risk/framingham.csv'
OUTPUT_DIR = '/content/drive/MyDrive/smoking-health-risk'
# =====================================================

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset tidak ditemukan di: {DATA_PATH}\n"
        "Solusi:\n"
        "1. Pastikan Google Drive sudah ter-mount (jalankan cell di atas).\n"
        "2. Upload file 'framingham.csv' ke Google Drive.\n"
        "3. Sesuaikan nilai DATA_PATH dengan lokasi file tersebut."
    )
print("Dataset ditemukan:", DATA_PATH)

## C2. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score
)
from sklearn.model_selection import GridSearchCV, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

RANDOM_STATE = 42
TEST_SIZE = 0.2

## C3. Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)

# --- validasi sederhana: pastikan kolom penting tersedia ---
TARGET = 'TenYearCHD'
NUMERIC_FEATURES = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
CATEGORICAL_FEATURES = ['male', 'education', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'currentSmoker']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

missing_cols = [c for c in ALL_FEATURES + [TARGET] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom berikut tidak ditemukan di dataset: {missing_cols}")
if not pd.api.types.is_numeric_dtype(df[TARGET]):
    raise ValueError("Kolom target 'TenYearCHD' harus berupa numerik (0/1).")

print(f"Dataset valid: {df.shape[0]} baris x {df.shape[1]} kolom")
df.head()

In [ ]:
df.info()

In [ ]:
# cek missing values
df.isnull().sum()

In [ ]:
# cek duplikat
df.duplicated().sum()

# D. Data Understanding

Dataset Framingham berisi 15 fitur klinis/demografis untuk memprediksi kejadian CHD dalam 10 tahun. Target `TenYearCHD` bersifat *imbalanced* (~85% kelas 0 vs ~15% kelas 1), sehingga ROC-AUC digunakan sebagai acuan utama evaluasi.

In [ ]:
df[TARGET].value_counts()

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x=df[TARGET])
plt.title('Distribusi Kelas Target (TenYearCHD)')
plt.xlabel('Kelas (0 = risiko rendah, 1 = risiko tinggi)')
plt.ylabel('Jumlah')
plt.show()

# E. Exploratory Data Analysis

In [ ]:
# distribusi usia
plt.figure(figsize=(6, 4))
sns.histplot(df['age'].dropna(), bins=30, kde=True)
plt.title('Distribusi Usia')
plt.xlabel('Usia (tahun)')
plt.show()

In [ ]:
# distribusi jumlah rokok per hari
plt.figure(figsize=(6, 4))
sns.histplot(df['cigsPerDay'].dropna(), bins=30, kde=True, color='orange')
plt.title('Distribusi Jumlah Rokok per Hari')
plt.xlabel('Rokok per hari')
plt.show()

In [ ]:
# hubungan kebiasaan merokok dengan target
smoke_risk = df.groupby('currentSmoker')[TARGET].mean()
plt.figure(figsize=(5, 4))
sns.barplot(x=['Tidak Merokok', 'Merokok'], y=smoke_risk.values)
plt.title('Proporsi Risiko Tinggi (CHD 10 Tahun) Berdasarkan Status Merokok')
plt.ylabel('Proporsi Risiko Tinggi')
plt.show()

In [ ]:
# korelasi fitur numerik terhadap target
plt.figure(figsize=(9, 7))
corr = df[NUMERIC_FEATURES + [TARGET]].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', square=True)
plt.title('Korelasi Fitur Numerik & Target')
plt.show()

# F. Preprocessing

## F1. Tentukan fitur dan label

- Label: `TenYearCHD` (0/1)
- Fitur: 15 kolom demografis, klinis, dan kebiasaan merokok
- Tidak ada fitur yang secara langsung membocorkan target (tidak ada kolom hasil laboratorium masa depan)

In [ ]:
fitur_df = df[ALL_FEATURES].copy()
label_df = df[TARGET].copy()

print("Jumlah fitur:", fitur_df.shape[1])
print("Distribusi label:")
print(label_df.value_counts())

## F2. Train/test split

Dataset dibagi 80% train dan 20% test dengan `stratify` agar proporsi kelas tetap seimbang. Test set **tidak disentuh** sampai evaluasi final untuk mencegah data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    fitur_df, label_df, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=label_df
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## F3. Bangun preprocessing pipeline

- **Numerik**: imputasi median + StandardScaler
- **Kategorikal**: imputasi most_frequent + OneHotEncoder (`drop='if_binary'`)

Preprocessing dipasang di dalam `Pipeline` sklearn sehingga hanya di-fit pada data latih.

In [ ]:
def build_preprocessor():
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore')),
    ])
    return ColumnTransformer([
        ('num', numeric_pipe, NUMERIC_FEATURES),
        ('cat', categorical_pipe, CATEGORICAL_FEATURES),
    ])

# G. Model Training & Comparison (Baseline)

Baseline: 3 model dilatih dengan hyperparameter default. Evaluasi awal memakai threshold 0.5 untuk memahami performa sebelum tuning.

In [ ]:
pipelines = {
    'logistic_regression': Pipeline([
        ('preprocess', build_preprocessor()),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    'random_forest': Pipeline([
        ('preprocess', build_preprocessor()),
        ('model', RandomForestClassifier(random_state=RANDOM_STATE)),
    ]),
    'gradient_boosting': Pipeline([
        ('preprocess', build_preprocessor()),
        ('model', GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
}

results = {}
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    results[name] = {'pipeline': pipeline, 'y_pred': y_pred, 'y_proba': y_proba}
    print(f'Training selesai: {name}')


## G1. Perbandingan model (default threshold 0.5)

In [ ]:
comparison = pd.DataFrame({
    name: {
        'accuracy': accuracy_score(y_test, res['y_pred']),
        'precision': precision_score(y_test, res['y_pred'], zero_division=0),
        'recall': recall_score(y_test, res['y_pred'], zero_division=0),
        'f1': f1_score(y_test, res['y_pred'], zero_division=0),
        'roc_auc': roc_auc_score(y_test, res['y_proba']),
    }
    for name, res in results.items()
}).T
comparison.round(4)

# H. Hyperparameter Tuning

Masing-masing model di-tuning menggunakan `GridSearchCV` dengan scoring **ROC-AUC** dan 5-fold CV pada train set. Setelah tuning, dilakukan **threshold optimization** (maksimum F1 dari out-of-fold CV) agar model dapat mengatasi class imbalance.

In [ ]:
param_grids = {
    'logistic_regression': {
        'model__C': [0.01, 0.1, 1.0, 10.0],
        'model__penalty': ['l2'],
        'model__class_weight': [None, 'balanced'],
    },
    'random_forest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [4, 8, 12, None],
        'model__min_samples_split': [2, 5, 10],
        'model__class_weight': [None, 'balanced'],
    },
    'gradient_boosting': {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__max_depth': [2, 3, 4],
    },
}

tuned = {}
for name, base_pipeline in pipelines.items():
    print(f'\n--- Tuning: {name} ---')
    gs = GridSearchCV(
        base_pipeline, param_grids[name],
        scoring='roc_auc', cv=5, n_jobs=-1, refit=True, verbose=0
    )
    gs.fit(X_train, y_train)
    tuned[name] = gs
    print(f'  Best CV ROC-AUC: {gs.best_score_:.4f}')
    print(f'  Best params: {gs.best_params_}')

In [ ]:
def find_best_threshold(pipeline):
    """Cari threshold yang memaksimalkan F1 pada out-of-fold CV train set."""
    oof_proba = cross_val_predict(
        pipeline, X_train, y_train, cv=5, method='predict_proba', n_jobs=-1
    )[:, 1]
    best_t, best_f1 = 0.5, -1.0
    for t in np.arange(0.05, 0.96, 0.025):
        pred = (oof_proba >= t).astype(int)
        f1 = f1_score(y_train, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(round(best_t, 3))

thresholds = {name: find_best_threshold(gs.best_estimator_) for name, gs in tuned.items()}
print('Threshold optimal (dari train CV):')
for name, t in thresholds.items():
    print(f'  {name}: {t}')

# I. Final Model Evaluation

Model terbaik dipilih berdasarkan **ROC-AUC test set** (metric threshold-independent), lalu klasifikasi menggunakan threshold optimal.

In [ ]:
final_comparison = {}
final_predictions = {}

for name, gs in tuned.items():
    pipeline = gs.best_estimator_
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= thresholds[name]).astype(int)

    final_comparison[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'threshold': thresholds[name],
        'cv_roc_auc': gs.best_score_,
    }
    final_predictions[name] = (pipeline, y_pred, y_proba)

final_df = pd.DataFrame(final_comparison).T
print('\n===== Perbandingan Model Final (Test Set) =====')
print(final_df.round(4).to_string())

best_name = final_df['roc_auc'].idxmax()
best_pipeline, y_pred_best, y_proba_best = final_predictions[best_name]
best_metrics = final_comparison[best_name]

print(f'\nModel terbaik: {best_name}')
print(f"  ROC-AUC : {best_metrics['roc_auc']:.4f}")
print(f"  Accuracy: {best_metrics['accuracy']:.4f}")
print(f"  Precision: {best_metrics['precision']:.4f}")
print(f"  Recall  : {best_metrics['recall']:.4f}")
print(f"  F1      : {best_metrics['f1']:.4f}")

## I1. Confusion matrix model terbaik

In [ ]:
cm = confusion_matrix(y_test, y_pred_best, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Risiko Rendah', 'Risiko Tinggi'],
            yticklabels=['Risiko Rendah', 'Risiko Tinggi'])
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title(f'Confusion Matrix — {best_name}')
plt.show()

## I2. Classification report

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_best, target_names=['Risiko Rendah', 'Risiko Tinggi'], zero_division=0))

# J. Feature Importance / Explainability

In [ ]:
def get_feature_names(preprocessor):
    names = []
    numeric_pipe = preprocessor.named_transformers_['num']
    scaler = numeric_pipe.named_steps['scaler']
    if hasattr(scaler, 'get_feature_names_out'):
        names.extend(scaler.get_feature_names_out(NUMERIC_FEATURES))
    else:
        names.extend(NUMERIC_FEATURES)
    categorical_pipe = preprocessor.named_transformers_['cat']
    onehot = categorical_pipe.named_steps['onehot']
    names.extend(onehot.get_feature_names_out(CATEGORICAL_FEATURES))
    return list(names)

preprocessor = best_pipeline.named_steps['preprocess']
model = best_pipeline.named_steps['model']
feature_names = get_feature_names(preprocessor)

if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
elif hasattr(model, 'coef_'):
    importances = np.abs(model.coef_[0])
else:
    importances = None

if importances is not None:
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)
    plt.figure(figsize=(7, 5))
    sns.barplot(x=fi.values, y=fi.index, hue=fi.index, palette='viridis', legend=False)
    plt.title(f'Feature Importance — {best_name}')
    plt.xlabel('Importance')
    plt.ylabel('Fitur')
    plt.tight_layout()
    plt.show()

> Interpretasi: feature importance menunjukkan fitur yang paling **berkontribusi terhadap prediksi model**, bukan hubungan sebab-akibat.

# K. Save Final Model & Hasil Prediksi

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# simpan model terbaik (pipeline utuh: preprocessing + model)
model_path = os.path.join(OUTPUT_DIR, 'model_colab.joblib')
joblib.dump(best_pipeline, model_path)
print('Model tersimpan di:', model_path)

# simpan fitur + prediksi + label asli dari data uji
hasil_lengkap = pd.concat([
    X_test.reset_index(drop=True),
    pd.DataFrame({'prediksi': y_pred_best, 'label_asli': y_test.reset_index(drop=True)}),
], axis=1)
hasil_path = os.path.join(OUTPUT_DIR, 'hasil_prediksi_colab.csv')
hasil_lengkap.to_csv(hasil_path, index=False)
print('Hasil prediksi tersimpan di:', hasil_path)

# sanity check: load ulang model dan pastikan prediksinya konsisten
reloaded = joblib.load(model_path)
assert np.array_equal(reloaded.predict(X_test.head(5)), best_pipeline.predict(X_test.head(5)))
print('Reload check: OK')

# L. Contoh Prediksi Satu Orang

In [ ]:
sample = pd.DataFrame([{
    'age': 52, 'male': 1, 'education': 2, 'currentSmoker': 1, 'cigsPerDay': 30,
    'BPMeds': 1, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 1,
    'totChol': 260, 'sysBP': 160, 'diaBP': 95, 'BMI': 31, 'heartRate': 88, 'glucose': 130,
}])

risk_score = reloaded.predict_proba(sample)[0, 1]
pred_class = (risk_score >= thresholds[best_name]).astype(int)

print('Input: pria 52 tahun, perokok aktif 30 batang/hari, hipertensi + diabetes')
print(f'Probabilitas risiko CHD 10 tahun: {risk_score:.1%}')
print(f'Prediksi kelas: {pred_class} ({"Risiko Tinggi" if pred_class == 1 else "Risiko Rendah"})')
print('\nCatatan: hasil ini adalah estimasi statistik, bukan diagnosis medis.')

# M. Kesimpulan

1. Dataset Framingham Heart Study (4.240 partisipan) diproses menjadi klasifikasi biner risiko CHD 10 tahun (`TenYearCHD`).
2. Missing values ditangani dengan imputasi di dalam pipeline, lalu data dibagi stratified 80/20.
3. Tiga model dibandingkan dan di-tuning menggunakan GridSearchCV (scoring ROC-AUC, 5-fold CV).
4. Threshold klasifikasi dipilih dari out-of-fold CV untuk menyeimbangkan precision-recall pada data imbalanced.
5. Model terbaik dipilih berdasarkan ROC-AUC test set, lalu disimpan sebagai pipeline utuh (`model_colab.joblib`).
6. Hasil model adalah estimasi statistik risiko kesehatan, **bukan diagnosis medis**.